<div style="padding: 20px; background-color: #1e1e2f; color: #ffffff; border-radius: 12px; border-left: 6px solid #48bb78; margin-bottom: 20px; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">
    <h1 style="margin: 0; font-size: 2.2em; font-weight: 700; color: #48bb78; display: flex; align-items: center; gap: 10px;">
        🩺 MediVLM Google Colab Inference
    </h1>
    <p style="margin: 8px 0 0 0; color: #a0aec0; font-size: 1.1em; line-height: 1.5;">
        Test your trained Vision-Language Model directly in Google Colab. This notebook mounts Google Drive to fetch your model checkpoint, lets you upload a medical image directly from your computer, and runs single-image inference to generate a clinical radiology report.
    </p>
</div>

### 📋 Steps to run:
1. **Connect to a GPU runtime** (Runtime -> Change runtime type -> T4 GPU recommended for faster inference).
2. **Execute Cell 1** to clone the repository and install all required libraries.
3. **Execute Cell 2** to mount your Google Drive (where your trained checkpoint is stored).
4. **Execute Cell 3** to locate your checkpoint file.
5. **Execute Cell 4** to upload your chest X-ray image directly.
6. **Execute Cell 5** to load the model and generate your clinical report!

## 1. Setup Environment
Clone the MediVLM repository and install the required packages.

In [ ]:
# 1. Setup Environment
# Clone the repository and install all dependencies
import os
import sys

print("🚀 Cloning MediVLM repository...")
!git clone https://github.com/sonai-commits/MediVLM.git

# Change working directory to the cloned repository
%cd MediVLM
sys.path.append(os.getcwd())

print("\n📦 Installing requirements (this might take a minute)...")
!pip install -r requirements.txt
!pip install radgraph RaTEScore

print("\n✅ Setup complete! Environment is ready.")

## 2. Mount Google Drive (Optional)
Mount your Google Drive if your checkpoint is located there. If you uploaded the checkpoint directly to Colab sidebar, you can skip this step.

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

print("📂 Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("\n✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"\n⚠️ Drive mounting skipped or failed: {e}")

## 3. Locate your Model Checkpoint
Run the cell below to scan both your local Colab file storage (`/content`) and Google Drive for potential `.ckpt` or `.pt` files. You can copy the correct path and paste it into the parameter field.

In [ ]:
# Search for potential checkpoint files in /content and Google Drive
import glob
import os

print("🔍 Searching for checkpoint files (this may take a few seconds)...\n")
local_root = "/content"
drive_root = "/content/drive/MyDrive"

ckpt_files = []
# 1. Search locally in /content
if os.path.exists(local_root):
    ckpt_files += glob.glob(os.path.join(local_root, "*.ckpt")) + glob.glob(os.path.join(local_root, "*.pt"))

# 2. Search recursively in Google Drive
if os.path.exists(drive_root):
    ckpt_files += glob.glob(os.path.join(drive_root, "**/*.ckpt"), recursive=True) + \
                  glob.glob(os.path.join(drive_root, "**/*.pt"), recursive=True)

if ckpt_files:
    print(f"✨ Found {len(ckpt_files)} potential checkpoint(s):")
    for idx, path in enumerate(ckpt_files[:15]):
        print(f" [{idx}] {path}")
    print("\n💡 Tip: Copy the correct path from above and paste it below.")
else:
    print("⚠️ No checkpoint files (.ckpt/.pt) automatically found. Please enter your path manually below.")

# Paste your checkpoint path below
CHECKPOINT_PATH = "/content/mimic_sample.ckpt" #@param {type:"string"}
print(f"\nSelected Checkpoint Path: {CHECKPOINT_PATH}")

## 4. Upload your Medical Image directly
Click **Choose Files** below to upload the chest X-ray image you want to run inference on. This image is stored locally in the ephemeral Colab environment.

In [ ]:
# Upload image directly to Colab
from google.colab import files
from PIL import Image
import IPython.display as display

print("📤 Select and upload your medical image (e.g., .png, .jpg):\n")
uploaded = files.upload()

if uploaded:
    # Get the uploaded image path
    uploaded_image_path = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded image successfully: {uploaded_image_path}")
    
    # Display the uploaded image
    img = Image.open(uploaded_image_path)
    print("\n🖼️ Uploaded Chest X-Ray:")
    display.display(img.resize((300, 300)))
else:
    uploaded_image_path = ""
    print("\n❌ No image uploaded. Please upload a file to proceed.")

## 5. Load Model & Generate Report
Select the model configuration YAML file corresponding to the dataset the model was trained on, customize generation settings (like the number of beams), and run inference!

In [ ]:
# Run Inference
import torch
from PIL import Image
from medivlm.data.transforms import build_image_transform
from medivlm.models import MediVLM
from medivlm.utils import load_checkpoint, load_config
import os
from pathlib import Path

# 1. Interactive Inputs
CONFIG_FILE = "configs/mimic_cxr_sample.yaml" #@param ["configs/mimic_cxr.yaml", "configs/mimic_cxr_sample.yaml", "configs/iu_xray.yaml", "configs/casia_cxr.yaml"]
NUM_BEAMS = 4 #@param {type:"integer"}

# 2. Check Device & Paths
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

if not 'uploaded_image_path' in globals() or not uploaded_image_path:
    raise ValueError("❌ Please upload an image first using the cell above!")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"❌ Checkpoint file not found at: {CHECKPOINT_PATH}")

# 3. Load Configurations and Model
print(f"📖 Loading config from {CONFIG_FILE}...")
cfg = load_config(CONFIG_FILE)

print("🏗️ Initializing MediVLM model...")
model = MediVLM(cfg).to(device)

print(f"💾 Loading model checkpoint: {CHECKPOINT_PATH}...")
load_checkpoint(CHECKPOINT_PATH, model=model, map_location=device)
model.eval()
print("✅ Model successfully loaded!")

# 4. Preprocess Image
print("⚙️ Preprocessing image...")
transform = build_image_transform(cfg.image_encoder.image_size, train=False)
img = Image.open(uploaded_image_path).convert("RGB")
image_tensor = transform(img).unsqueeze(0).to(device)

# 5. Run Generation
print("🔮 Generating clinical report (this may take a few seconds)...\n")
with torch.no_grad():
    reports = model.generate(image_tensor, num_beams=NUM_BEAMS)
report = reports[0]

# 6. Display Beautiful Output
print("=" * 80)
print(f"📄 GENERATED CLINICAL RADIOLOGY REPORT FOR: {uploaded_image_path}")
print("=" * 80)
print(report)
print("=" * 80)